In [ ]:
import sys
import json
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

TRAIN_JSONL = ROOT / "train.jsonl"
MAX_LINES = 1090
MAX_REQUIREMENTS = 2000
DOC_ID = "train_jsonl"
CHECKPOINT_PATH = ROOT / "requirements_checkpoint.json"
EMBEDDINGS_CACHE_DIR = ROOT / "data"
LOAD_CHECKPOINT = True

In [ ]:
from src.schema import Entry

if LOAD_CHECKPOINT and CHECKPOINT_PATH.exists():
    data = json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
    requirements_with_tiers = [(Entry(**d["entry"]), d["tier"]) for d in data["requirements_with_tiers"]]
    trace_pairs = [tuple(p) for p in data["trace_pairs"]]
    requirements = [e for e, _ in requirements_with_tiers]
    DOC_ID = data.get("doc_id", DOC_ID)
    emb = data.get("embeddings") or []
    embeddings = emb if len(emb) == len(requirements_with_tiers) else None
    print(f"Loaded checkpoint: {len(requirements_with_tiers)} requirements, {len(trace_pairs)} trace pairs, embeddings={'yes' if embeddings else 'no'}.")
else:
    requirements = requirements_with_tiers = trace_pairs = embeddings = None

In [ ]:
from src.schema import Entry

def is_req_value(v):
    if v is True: return True
    if v is False or v is None: return False
    return str(v).strip().lower() in ("true", "yes", "1", "requirement")

entries_all = []
lines_read = 0
with open(TRAIN_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if lines_read >= MAX_LINES: break
        line = line.strip()
        if not line: continue
        try:
            row = json.loads(line)
            out = row.get("output") or {}
            for e in out.get("entries") or []:
                if not is_req_value(e.get("is_requirement")): continue
                entries_all.append(Entry(
                    doc_title=e.get("doc_title") or "",
                    page_number=int(e.get("page_number", 0)),
                    section_title=e.get("section_title") or "",
                    text=(e.get("text") or "").strip(),
                    is_requirement="true",
                ))
                if len(entries_all) >= MAX_REQUIREMENTS: break
        except (json.JSONDecodeError, KeyError, TypeError):
            continue
        lines_read += 1
        if len(entries_all) >= MAX_REQUIREMENTS: break

requirements = entries_all
print(f"Loaded {len(requirements)} requirements.")

In [ ]:
from src.nova_tier import assign_tiers_to_requirements_with_fallback

requirements_with_tiers = assign_tiers_to_requirements_with_fallback(requirements, default_tier="system")
for e, tier in requirements_with_tiers[:10]:
    print(f"  [{tier}] {e.text[:60]}...")
print(f"... total {len(requirements_with_tiers)}")

In [ ]:
from src.nova_traces import get_trace_pairs_from_nova_with_fallback

trace_pairs = get_trace_pairs_from_nova_with_fallback(requirements_with_tiers, batch_size=12)
print(f"Nova: {len(trace_pairs)} pairs.")

In [ ]:
from src.titan_embeddings import (
    get_embeddings_for_requirements,
    embeddings_cache_path,
)

embed_cache = embeddings_cache_path(ROOT, DOC_ID)
if embeddings is None or len(embeddings) != len(requirements_with_tiers):
    embeddings = get_embeddings_for_requirements(
        requirements_with_tiers,
        cache_path=embed_cache,
        doc_id=DOC_ID,
        dimensions=1024,
    )
else:
    print(f"Using {len(embeddings)} embeddings from checkpoint.")

In [ ]:
from src.neo4j_loader import (
    get_driver,
    load_tiered_requirements_into_neo4j,
    requirement_full_id,
    create_trace_edges,
    tier_based_trace_pairs,
    filter_trace_pairs_to_adjacent_tiers,
)

driver = get_driver()
n = load_tiered_requirements_into_neo4j(
    requirements_with_tiers, driver=driver, document_id=DOC_ID, embeddings=embeddings if embeddings else None
)
print(f"Loaded {n} requirement nodes.")

full_ids = [requirement_full_id(DOC_ID, e, i) for i, (e, _) in enumerate(requirements_with_tiers)]
tier_pairs = tier_based_trace_pairs(requirements_with_tiers)
combined = list({p for p in trace_pairs + tier_pairs})
all_pairs = filter_trace_pairs_to_adjacent_tiers(requirements_with_tiers, combined)
edges = create_trace_edges(full_ids, all_pairs, driver=driver)
print(f"Created {edges} TRACES_TO edges.")
driver.close()

In [ ]:

from pathlib import Path
if "CHECKPOINT_PATH" not in globals():
    CHECKPOINT_PATH = Path.cwd() / "requirements_checkpoint.json"
if "DOC_ID" not in globals():
    DOC_ID = "train_jsonl"

checkpoint = {
    "requirements_with_tiers": [{"entry": e.model_dump(), "tier": t} for e, t in requirements_with_tiers],
    "trace_pairs": [list(p) for p in trace_pairs],
    "doc_id": DOC_ID,
}
if embeddings and len(embeddings) == len(requirements_with_tiers):
    checkpoint["embeddings"] = embeddings
CHECKPOINT_PATH.write_text(json.dumps(checkpoint, indent=2), encoding="utf-8")
print(f"Saved checkpoint to {CHECKPOINT_PATH} ({len(requirements_with_tiers)} requirements, {len(trace_pairs)} pairs, embeddings={'yes' if checkpoint.get('embeddings') else 'no'}).")